# UIT LegalIR — RTX Pro 6000 offline
Attach the competition-data dataset and a dataset created by `build_offline_bundle.ipynb`. Set **Internet: Off** and use one RTX Pro 6000 GPU.

In [ ]:
import os
# Set these before importing Transformers, Sentence Transformers, or Hugging Face Hub.
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

from pathlib import Path
DATASET_DIR = Path('/kaggle/input/REPLACE_WITH_COMPETITION_DATASET_SLUG')
BUNDLE_DIR = Path('/kaggle/input/REPLACE_WITH_OFFLINE_BUNDLE_SLUG/legalir-offline-bundle')
WORK_DIR = Path('/kaggle/working/legalir-offline-run')

In [ ]:
import importlib.metadata
import json
import shutil
import subprocess
import sys

def run(*command, cwd=None, env=None):
    print('+', ' '.join(map(str, command)))
    subprocess.run(list(map(str, command)), cwd=cwd, env=env, check=True)

manifest_path = BUNDLE_DIR / 'manifests' / 'bundle_manifest.json'
if not manifest_path.is_file():
    raise FileNotFoundError(f'Missing offline bundle manifest: {manifest_path}')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
for model in manifest['models']:
    snapshot = BUNDLE_DIR / model['local_path']
    if not (snapshot / 'config.json').is_file():
        raise FileNotFoundError(f'Missing model snapshot: {snapshot}')
if not (BUNDLE_DIR / 'wheels').is_dir():
    raise FileNotFoundError('Offline wheelhouse is missing')

# Install the pinned lightweight NLP stack from the bundle. This leaves Kaggle's
# tested NumPy/SciPy/scikit-learn/CUDA torch stack unchanged.
run(sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--find-links', BUNDLE_DIR / 'wheels', '-r', BUNDLE_DIR / 'requirements-offline.txt')
nlp_versions = {name: importlib.metadata.version(name) for name in ('sentence-transformers', 'transformers', 'tokenizers')}
print('Pinned NLP runtime:', nlp_versions)
project_wheels = sorted((BUNDLE_DIR / 'wheels').glob('uit_legalir-*.whl'))
if len(project_wheels) != 1:
    raise RuntimeError(f'Expected exactly one uit_legalir wheel, found {project_wheels}')
run(sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--force-reinstall', project_wheels[0])
# Do not run global `pip check`: Kaggle base-image packages outside LegalIR can
# have conflicting optional constraints. The following model preflight validates
# the actual installed LegalIR runtime without accessing the network.
print('Offline bundle project commit:', manifest['project_commit'])

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is available. Select RTX Pro 6000 before running this notebook.')
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda, 'capability:', torch.cuda.get_device_capability(0))
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision('high')

contexts_source = DATASET_DIR / 'selected-contexts' / 'selected-contexts'
if not contexts_source.is_dir():
    raise FileNotFoundError(f'Missing nested competition corpus directory: {contexts_source}')
context_count = sum(1 for _ in contexts_source.glob('context_*.json'))
if not context_count:
    raise FileNotFoundError(f'No context_*.json files found in {contexts_source}')
for filename in ('train.json', 'public-official.json'):
    if not (DATASET_DIR / filename).is_file():
        raise FileNotFoundError(f'Missing competition input: {DATASET_DIR / filename}')

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True)
(WORK_DIR / 'selected-contexts').symlink_to(contexts_source, target_is_directory=True)
for filename in ('train.json', 'public-official.json'):
    (WORK_DIR / filename).symlink_to(DATASET_DIR / filename)
print(f'Using {context_count} legal contexts from {contexts_source}')

In [ ]:
import yaml

config = yaml.safe_load((BUNDLE_DIR / 'configs' / 'kaggle_rtx_pro_6000.yaml').read_text(encoding='utf-8'))
# Model names in the manifest and config are deliberately identical.
for name in config['models']:
    config['models'][name]['local_path'] = str(BUNDLE_DIR / 'models' / name)
    config['models'][name]['local_files_only'] = True
config_path = WORK_DIR / 'kaggle_rtx_pro_6000.yaml'
config_path.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding='utf-8')
print(config_path.read_text(encoding='utf-8'))

In [ ]:
# Model preflight proves every snapshot, including Jina custom code, runs with Hub disabled.
from legalir.embeddings import load_encoder
from legalir.rerank import JinaListwiseReranker, VietnamesePairwiseReranker

for name in ('vietlegal_e5', 'vietnamese_embedding', 'nemotron'):
    model = load_encoder(config['models'][name], config['runtime'])
    assert len(model.encode(['kiểm tra offline'], convert_to_numpy=True)) == 1
    del model
    torch.cuda.empty_cache()
pairwise = VietnamesePairwiseReranker(config)
assert len(pairwise.rank('câu hỏi', ['văn bản một', 'văn bản hai'])) == 2
pairwise.close()
jina = JinaListwiseReranker(config)
assert len(jina.rank('câu hỏi', ['văn bản một', 'văn bản hai'])) == 2
jina.close()
print('All local offline model preflight tests passed.')

In [ ]:
# One RTX Pro 6000: run model stages sequentially; do not start competing CUDA processes.
base = [sys.executable, '-m', 'legalir']
def legalir(*args):
    run(*base, *args, cwd=WORK_DIR, env=os.environ.copy())

legalir('prepare', '--config', config_path, '--resume')
legalir('audit', '--config', config_path)
legalir('index', '--config', config_path, '--lexical-only', '--resume')
for model in ('vietlegal_e5', 'vietnamese_embedding', 'nemotron'):
    legalir('index', '--config', config_path, '--model', model, '--resume')
legalir('tune', '--config', config_path, '--resume')
for engine in ('vietnamese_reranker', 'jina'):
    legalir('rerank', '--config', config_path, '--split', 'train', '--fold', '0', '--engine', engine, '--resume')
legalir('rerank', '--config', config_path, '--split', 'train', '--fold', '0', '--resume')
legalir('tune', '--config', config_path, '--final', '--fold', '0', '--resume')
legalir('retrieve', '--config', config_path, '--split', 'public', '--resume')
for engine in ('vietnamese_reranker', 'jina'):
    legalir('rerank', '--config', config_path, '--split', 'public', '--engine', engine, '--resume')
legalir('predict', '--config', config_path, '--output', WORK_DIR / 'submission.json', '--resume')
run('zip', '-j', WORK_DIR / 'submission.zip', WORK_DIR / 'submission.json')
print('Submission:', WORK_DIR / 'submission.zip')